In [ ]:
!pip install kaggle==1.5.16

In [ ]:
! chmod 600 .kaggle/kaggle.json

In [ ]:
! kaggle competitions download Walmart-Recruiting-Store-Sales-Forecasting

In [ ]:
! unzip Walmart-Recruiting-Store-Sales-Forecasting.zip

In [ ]:
! unzip features.csv.zip
! unzip test.csv.zip
! unzip train.csv.zip

# DLinear — Walmart Store Sales Forecasting

In [ ]:
!pip install "numpy<2" "torchvision==0.17.0" "torch==2.2.0" "neuralforecast==1.7.4" optuna mlflow dagshub wandb -q --force-reinstall

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle, os
import wandb
import mlflow
import mlflow.pyfunc

from neuralforecast import NeuralForecast
from neuralforecast.models import DLinear
from neuralforecast.auto import AutoDLinear
from neuralforecast.losses.pytorch import MAE
from ray import tune

WANDB_ENTITY  = 'ikvas22-free-university-of-tbilisi'
WANDB_PROJECT = 'Walmart Weekly Sales Prediction'
WANDB_GROUP   = 'DLinear'

MLFLOW_EXPERIMENT = 'DLinear_Training'
MLFLOW_MODEL_NAME = 'dlinear_walmart_best'

TRAIN_PATH    = 'train.csv'
FEATURES_PATH = 'features.csv'
STORES_PATH   = 'stores.csv'

H          = 4
INPUT_SIZE = 52
N_TRIALS   = 5
VAL_START  = '2012-04-01'

wandb.login()

print('Setup OK')

## 1. Pre-processing

In [ ]:
run = wandb.init(
    entity   = WANDB_ENTITY,
    project  = WANDB_PROJECT,
    group    = WANDB_GROUP,
    job_type = 'preprocessing',
    name     = 'DLinear_Preprocessing',
)

train_raw    = pd.read_csv(TRAIN_PATH,    parse_dates=['Date'])
features_raw = pd.read_csv(FEATURES_PATH, parse_dates=['Date'])
stores_raw   = pd.read_csv(STORES_PATH)

df = (
    train_raw
    .merge(features_raw, on=['Store', 'Date', 'IsHoliday'], how='left')
    .merge(stores_raw,   on=['Store'],                       how='left')
)

wandb.log({
    'raw_rows' : df.shape[0],
    'raw_cols' : df.shape[1],
    'stores'   : df['Store'].nunique(),
    'depts'    : df['Dept'].nunique(),
    'date_min' : str(df['Date'].min().date()),
    'date_max' : str(df['Date'].max().date()),
})

null_pct = df.isnull().mean().mul(100).round(2)
null_df  = null_pct[null_pct > 0].reset_index()
null_df.columns = ['column', 'null_pct']
wandb.log({'null_percentages': wandb.Table(dataframe=null_df)})
print('Nulls (%):')
print(null_df.to_string(index=False))

# DLinear only needs the raw sales series — no tabular features
df['unique_id'] = df['Store'].astype(str) + '_' + df['Dept'].astype(str)
df_nf = (
    df[['unique_id', 'Date', 'Weekly_Sales', 'IsHoliday']]
    .rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
    .sort_values(['unique_id', 'ds'])
    .reset_index(drop=True)
)

min_len    = INPUT_SIZE + H
series_len = df_nf.groupby('unique_id')['ds'].count()
valid_ids  = series_len[series_len >= min_len].index
dropped    = series_len[series_len < min_len].index.tolist()
df_nf      = df_nf[df_nf['unique_id'].isin(valid_ids)].reset_index(drop=True)

wandb.log({
    'total_series'   : len(series_len),
    'valid_series'   : len(valid_ids),
    'dropped_series' : len(dropped),
    'dropped_ids'    : dropped,
    'min_series_len' : int(series_len[valid_ids].min()),
    'max_series_len' : int(series_len[valid_ids].max()),
})

print(f'\nValid series : {len(valid_ids)}')
print(f'Dropped      : {len(dropped)} (shorter than {min_len} weeks)')

df_train = df_nf[df_nf['ds'] <  VAL_START].copy()
df_val   = df_nf[df_nf['ds'] >= VAL_START].copy()

wandb.log({
    'train_rows'     : len(df_train),
    'val_rows'       : len(df_val),
    'train_date_min' : str(df_train['ds'].min().date()),
    'train_date_max' : str(df_train['ds'].max().date()),
    'val_date_min'   : str(df_val['ds'].min().date()),
    'val_date_max'   : str(df_val['ds'].max().date()),
    'val_start'      : VAL_START,
    'horizon_weeks'  : H,
    'lookback_weeks' : INPUT_SIZE,
})

print(f'Train : {df_train["ds"].min().date()} → {df_train["ds"].max().date()}  ({len(df_train):,} rows)')
print(f'Val   : {df_val["ds"].min().date()} → {df_val["ds"].max().date()}  ({len(df_val):,} rows)')

wandb.finish()

## 2. Training

In [ ]:
def wmae(y_true: np.ndarray, y_pred: np.ndarray, is_holiday: np.ndarray) -> float:
    weights = np.where(is_holiday, 5.0, 1.0)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))


def evaluate(nf: NeuralForecast, pred_col: str) -> tuple:
    preds   = nf.predict()
    eval_df = df_val.merge(
        preds.rename(columns={pred_col: 'y_pred'}),
        on=['unique_id', 'ds'], how='inner'
    )
    score_wmae = wmae(eval_df['y'].values, eval_df['y_pred'].values, eval_df['IsHoliday'].values)
    score_mae  = float(np.mean(np.abs(eval_df['y'].values - eval_df['y_pred'].values)))
    return score_wmae, score_mae, eval_df

In [ ]:
# ── Baseline: single DLinear with sensible defaults ────────────
# DLinear is a very lightweight model — decomposition + 2 linear layers.
# It often beats complex Transformers on trend-dominated series.
run_baseline = wandb.init(
    entity   = WANDB_ENTITY,
    project  = WANDB_PROJECT,
    group    = WANDB_GROUP,
    job_type = 'training',
    name     = 'DLinear_Baseline',
    config   = {
        'input_size'      : INPUT_SIZE,
        'h'               : H,
        'moving_avg_window': 25,
        'max_steps'       : 500,
        'learning_rate'   : 1e-3,
        'batch_size'      : 32,
        'loss'            : 'MAE',
    }
)

baseline_model = DLinear(
    h                  = H,
    input_size         = INPUT_SIZE,
    moving_avg_window  = 25,
    loss               = MAE(),
    max_steps          = 500,
    learning_rate      = 1e-3,
    batch_size         = 32,
    val_check_steps    = 50,
    logger             = True,
)

nf_baseline = NeuralForecast(models=[baseline_model], freq='W-FRI')
nf_baseline.fit(df=df_train, val_size=len(df_val['ds'].unique()))

baseline_wmae, baseline_mae, eval_baseline = evaluate(nf_baseline, 'DLinear')

wandb.log({
    'val_wmae' : baseline_wmae,
    'val_mae'  : baseline_mae,
})

print(f'Baseline WMAE : {baseline_wmae:,.2f}')
print(f'Baseline MAE  : {baseline_mae:,.2f}')

wandb.finish()

In [ ]:
# ── AutoDLinear: hyperparameter search ─────────────────────────
# DLinear has fewer hyperparameters than N-BEATS so search is faster.
# Key parameter is moving_avg_window — controls how trend is extracted.
run_search = wandb.init(
    entity   = WANDB_ENTITY,
    project  = WANDB_PROJECT,
    group    = WANDB_GROUP,
    job_type = 'training',
    name     = 'DLinear_HPSearch',
    config   = {
        'n_trials'    : N_TRIALS,
        'h'           : H,
        'search_space': {
            'input_size'        : [26, 52, 104],
            'moving_avg_window' : '4–52',
            'learning_rate'     : '1e-4 – 1e-3 (log)',
            'batch_size'        : [16, 32, 64],
        }
    }
)

dlinear_config = {
    'input_size'        : tune.choice([26, 52, 104]),
    'moving_avg_window' : tune.choice([4, 8, 13, 25, 52]),
    'learning_rate'     : tune.loguniform(1e-4, 1e-3),
    'batch_size'        : tune.choice([16, 32, 64]),
    'max_steps'         : 500,
}

auto_model = AutoDLinear(
    h           = H,
    config      = dlinear_config,
    num_samples = N_TRIALS,
    loss        = MAE(),
)

nf_auto = NeuralForecast(models=[auto_model], freq='W-FRI')
nf_auto.fit(df=df_train, val_size=len(df_val['ds'].unique()))

best_wmae, best_mae, eval_best = evaluate(nf_auto, 'AutoDLinear')
best_params = auto_model.results.get_best_result().config
trials_df   = auto_model.results.get_dataframe()

wandb.log({
    'best_val_wmae'    : best_wmae,
    'best_val_mae'     : best_mae,
    'baseline_wmae'    : baseline_wmae,
    'wmae_improvement' : baseline_wmae - best_wmae,
    'best_params'      : str(best_params),
    'trials_completed' : N_TRIALS,
    'all_trials'       : wandb.Table(dataframe=trials_df),
})

print(f'Best WMAE     : {best_wmae:,.2f}')
print(f'Baseline WMAE : {baseline_wmae:,.2f}')
print(f'Improvement   : {baseline_wmae - best_wmae:,.2f}')
print(f'Best params   : {best_params}')

# Prediction plots
sample_ids = eval_best['unique_id'].unique()[:6]
fig, axes  = plt.subplots(3, 2, figsize=(14, 10))
axes       = axes.flatten()

for ax, uid in zip(axes, sample_ids):
    history = df_train[df_train['unique_id'] == uid].tail(52)
    actual  = eval_best[eval_best['unique_id'] == uid]
    ax.plot(history['ds'], history['y'],     label='History', color='steelblue')
    ax.plot(actual['ds'],  actual['y'],      label='Actual',  color='black')
    ax.plot(actual['ds'],  actual['y_pred'], label='DLinear', color='tomato', linestyle='--')
    hol = actual[actual['IsHoliday'] == 1]
    ax.scatter(hol['ds'], hol['y'], color='gold', zorder=5, s=40, label='Holiday')
    ax.set_title(f'Store-Dept {uid}', fontsize=10)
    ax.legend(fontsize=7)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('DLinear Best Model — Validation Predictions', fontsize=13)
plt.tight_layout()
wandb.log({'prediction_plots': wandb.Image(fig)})
plt.show()

per_series = (
    eval_best
    .groupby('unique_id')
    .apply(lambda g: wmae(g['y'].values, g['y_pred'].values, g['IsHoliday'].values))
    .reset_index()
    .rename(columns={0: 'wmae'})
    .sort_values('wmae', ascending=False)
)
wandb.log({'per_series_wmae': wandb.Table(dataframe=per_series)})

wandb.finish()

## 3. Save Best Model to MLflow Registry

In [ ]:
import dagshub
dagshub.init(repo_owner='ikvas22', repo_name='Walmart-Recruiting---Store-Sales-Forecasting', mlflow=True)

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT)

In [ ]:
class DLinearWrapper(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        with open(context.artifacts['nf_model'], 'rb') as f:
            self.nf = pickle.load(f)

    def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
        """
        Accepts raw DataFrame with [Store, Dept, Date, Weekly_Sales].
        Returns [Store, Dept, Date, Weekly_Sales_pred].
        """
        df_in              = model_input.copy()
        df_in['Date']      = pd.to_datetime(df_in['Date'])
        df_in['unique_id'] = df_in['Store'].astype(str) + '_' + df_in['Dept'].astype(str)
        df_nf_in = (
            df_in
            .rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
            [['unique_id', 'ds', 'y']]
            .sort_values(['unique_id', 'ds'])
        )
        preds    = self.nf.predict(df=df_nf_in)
        pred_col = [c for c in preds.columns if c not in ['unique_id', 'ds']][0]
        preds    = preds.rename(columns={pred_col: 'Weekly_Sales_pred'})
        preds[['Store', 'Dept']] = preds['unique_id'].str.split('_', expand=True).astype(int)
        return preds[['Store', 'Dept', 'ds', 'Weekly_Sales_pred']].rename(columns={'ds': 'Date'})


os.makedirs('mlflow_artifacts', exist_ok=True)
model_path = 'mlflow_artifacts/nf_auto_dlinear.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(nf_auto, f)

with mlflow.start_run(run_name='DLinear_Best_Model') as run:
    mlflow.log_params(best_params)
    mlflow.log_metrics({
        'val_wmae'         : best_wmae,
        'val_mae'          : best_mae,
        'baseline_wmae'    : baseline_wmae,
        'wmae_improvement' : baseline_wmae - best_wmae,
        'n_trials'         : N_TRIALS,
    })
    mlflow.pyfunc.log_model(
        artifact_path         = 'dlinear_model',
        python_model          = DLinearWrapper(),
        artifacts             = {'nf_model': model_path},
        registered_model_name = MLFLOW_MODEL_NAME,
    )
    print(f'Registered: {MLFLOW_MODEL_NAME}')
    print(f'Run ID    : {run.info.run_id}')
    print(f'Best WMAE : {best_wmae:,.2f}')

In [ ]:
loaded = mlflow.pyfunc.load_model(f'models:/{MLFLOW_MODEL_NAME}/latest')
sample = train_raw[train_raw['Store'] == 1][['Store', 'Dept', 'Date', 'Weekly_Sales']].head(100)
result = loaded.predict(sample)
print('Registry load OK')
print(result.head())